In [3]:
import os
import urllib.request

# 1. Localizar la carpeta que PANNs está buscando en tu usuario de Windows
panns_folder = os.path.join(os.path.expanduser('~'), 'panns_data')
os.makedirs(panns_folder, exist_ok=True)

csv_path = os.path.join(panns_folder, 'class_labels_indices.csv')

# 2. Descargar el archivo manualmente usando Python (sin depender de wget de Linux)
if not os.path.exists(csv_path):
    print(f"Descargando archivo faltante de AudioSet en: {csv_path}")
    url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
    urllib.request.urlretrieve(url, csv_path)
    print("¡Descarga completada! Ya hemos arreglado el bug de panns_inference.")
else:
    print("El archivo ya existe. Todo listo.")

Descargando archivo faltante de AudioSet en: C:\Users\Usuario\panns_data\class_labels_indices.csv
¡Descarga completada! Ya hemos arreglado el bug de panns_inference.


In [5]:
import os
import urllib.request

# 1. Localizar la carpeta panns_data de tu usuario
panns_folder = os.path.join(os.path.expanduser('~'), 'panns_data')
os.makedirs(panns_folder, exist_ok=True)

pth_path = os.path.join(panns_folder, 'Cnn14_mAP=0.431.pth')

# 2. Descargar los pesos de PANNs manualmente
if not os.path.exists(pth_path):
    print(f"Descargando pesos de PANNs (CNN14) en: {pth_path}")
    print("Esto puede tardar unos minutos (~300MB), no lo canceles...")
    
    # URL oficial de Zenodo para los pesos de PANNs
    url = "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1"
    urllib.request.urlretrieve(url, pth_path)
    
    print("¡Descarga de pesos completada con éxito!")
else:
    print("El archivo de pesos ya existe. Todo listo.")

Descargando pesos de PANNs (CNN14) en: C:\Users\Usuario\panns_data\Cnn14_mAP=0.431.pth
Esto puede tardar unos minutos (~300MB), no lo canceles...
¡Descarga de pesos completada con éxito!


In [6]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from pathlib import Path
from panns_inference import AudioTagging

# ── 1. CONFIGURACIÓN Y RUTAS ──
PROJECT_ROOT = Path.cwd().parent
DATASET_TSV = PROJECT_ROOT / 'mtg-jamendo-dataset' / 'data' / 'autotagging_moodtheme.tsv'
AUDIO_DIR = PROJECT_ROOT / 'data' / 'audio' 
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'embeddings_fase3'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 2. CARGAR DATASET ──
print("Cargando dataset...")
registros = []
with open(DATASET_TSV, 'r', encoding='utf-8') as f:
    next(f)
    for linea in f:
        if not linea.strip(): continue
        columnas = linea.strip().split('\t')
        if len(columnas) >= 6:
            track_id = columnas[0].replace('track_', '').lstrip('0') 
            if track_id == '': track_id = '0'
            registros.append({
                'track_id': track_id,
                'path': columnas[3]
            })

df_completo = pd.DataFrame(registros)
print(f"Total de canciones en índice: {len(df_completo)}")

# ── 3. CONFIGURACIÓN DEL MODELO PANNs (Rama Acústica) ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando dispositivo: {device}")

# Cargamos el modelo CNN14 de PANNs (pre-entrenado en AudioSet)
# La primera vez descargará el archivo .pth automáticamente
print("Cargando modelo pre-entrenado PANNs (CNN14)...")
model = AudioTagging(checkpoint_path=None, device=device)

# ── 4. EXTRACCIÓN TEMPORAL (30 SEGUNDOS -> 30 EMBEDDINGS) ──
panns_temporal_embeddings = []
track_ids_validos = []

print("Iniciando extracción de secuencias acústicas (30 frames/audio)...")

# Parámetros de la ventana para PANNs
duracion_total = 30.0
ventana_segundos = 1.0
sr = 32000 # PANNs requiere estrictamente 32kHz
muestras_por_ventana = int(sr * ventana_segundos)

for idx, row in tqdm(df_completo.iterrows(), total=len(df_completo)):
    track_id = str(row['track_id'])
    
    # Lógica de rutas para fallback a .low.mp3
    path_limpio = str(row['path']).strip() 
    audio_path_normal = AUDIO_DIR / path_limpio
    audio_path_low = AUDIO_DIR / path_limpio.replace('.mp3', '.low.mp3')

    if os.path.exists(audio_path_normal):
        audio_path = audio_path_normal
    elif os.path.exists(audio_path_low):
        audio_path = audio_path_low
    else:
        continue 
        
    try:
        # 1. Cargar exactamente 30 segundos a 32kHz
        audio_array, _ = librosa.load(str(audio_path), sr=sr, duration=duracion_total)
        
        # 2. Padding por si dura menos de 30s
        if len(audio_array) < int(sr * duracion_total):
            pad_length = int(sr * duracion_total) - len(audio_array)
            audio_array = np.pad(audio_array, (0, pad_length), mode='constant')
            
        # 3. Trocear el audio en 30 fragmentos (Shape: 30 x 32000)
        chunks = [audio_array[i:i + muestras_por_ventana] for i in range(0, len(audio_array), muestras_por_ventana)]
        
        if len(chunks[-1]) < muestras_por_ventana:
            chunks = chunks[:-1]
            
        # Convertir a matriz numpy para la librería panns_inference
        chunks_np = np.array(chunks) 
        
        # 4. Inferencia en batch. 
        # panns_inference devuelve: (predicciones_audioset, embeddings)
        # Nos quedamos solo con los embeddings latentes
        _, embeddings = model.inference(chunks_np)
        
        # Guardamos la secuencia (Shape: 30, 2048)
        panns_temporal_embeddings.append(embeddings)
        track_ids_validos.append(track_id)
            
    except Exception as e:
        pass # Ignorar audios corruptos

# ── 5. GUARDAR MATRIZ TRIDIMENSIONAL ──
print("\n Guardando resultados temporales de PANNs...")

np.save(OUTPUT_DIR / 'panns_temporal_embeddings.npy', np.array(panns_temporal_embeddings))
np.save(OUTPUT_DIR / 'track_ids_panns_temporal.npy', np.array(track_ids_validos))

print(f"¡Completado! Se extrajeron las secuencias de {len(track_ids_validos)} canciones.")
print(f"Dimensiones de la matriz final: {np.array(panns_temporal_embeddings).shape}")

Cargando dataset...
Total de canciones en índice: 18486
Usando dispositivo: cuda
Cargando modelo pre-entrenado PANNs (CNN14)...
Checkpoint path: C:\Users\Usuario/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1
Iniciando extracción de secuencias acústicas (30 frames/audio)...


100%|██████████| 18486/18486 [16:02<00:00, 19.20it/s]



 Guardando resultados temporales de PANNs...
¡Completado! Se extrajeron las secuencias de 18486 canciones.
Dimensiones de la matriz final: (18486, 30, 2048)
